# Advanced Scalable PIR Pipeline (V2.2)

This notebook implements the Two-Stage Recommender Pipeline optimized for massive scale.
It incorporates advanced Memory Optimization, extracts highly dense features, and correctly references `updated_date` and `size_to_age` mappings.


In [9]:
import polars as pl
import numpy as np
import lightgbm as lgb
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import gc
import warnings
import re
warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(10)


polars.config.Config

## 1. Item Size to Target Age Mapping
We use the rules from `size_to_age.py` to convert item 'size' strings into numerical target ages (in years).


In [10]:
def standardize_age(text):
    if text is None: return -1.0
    raw_text = str(text).strip()
    clean_text = raw_text.lower()
    
    if re.search(r'(\*|x\d|cm)', clean_text): return 0.5
    if re.search(r'\bb\d{2}\b', clean_text): return 25.0 # Adult proxy
    if 's17' in clean_text: return 1.0
    if '110' in clean_text: return 5.0
    if "không xác định" in clean_text or not clean_text: return -1.0
    
    diaper_map = {
        r'\bnb\b': 0, r'\bss\b': 0, r'\bsơ sinh\b': 0,
        r'\bs\b': 0.25, r'\bm\b': 0.6, r'\bl\b': 1.2,
        r'\bxl\b': 2.0, r'\bxxl\b': 3.5
    }
    for pattern, val in diaper_map.items():
        if re.search(pattern, clean_text): return float(val)
        
    range_match = re.search(r'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)', clean_text)
    if range_match:
        s, e = float(range_match.group(1)), float(range_match.group(2))
        avg = (s + e) / 2
        if any(x in clean_text for x in ['m', 'tháng']): return round(avg / 12, 3)
        return avg
        
    m_match = re.search(r'(\d+\.?\d*)\s*(m|tháng)', clean_text)
    if m_match: return round(float(m_match.group(1)) / 12, 3)
    
    y_match = re.search(r'(\d+\.?\d*)\s*(y|t|tuổi)', clean_text)
    if y_match: return float(y_match.group(1))
    
    pure_num = re.search(r'^(\d+)$', clean_text)
    if pure_num:
        val = float(pure_num.group(1))
        if val > 6: return round(val/12, 3)
        else: return val
        
    return -1.0


## 2. Data Loading and Memory Optimization
We load transactions, properly reading `updated_date`, `location`, and `price`.


In [11]:
def load_and_prep_data(transaction_path):
    df = pl.scan_parquet(transaction_path).select([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('item_id').cast(pl.Utf8), # Keep as Utf8 to avoid Categorical cache mismatches during concat
        pl.col('updated_date').cast(pl.Datetime).alias('event_ts'),
        pl.col('quantity').cast(pl.Float32).fill_null(0.0),
        pl.col('event_type').cast(pl.Categorical),
        pl.col('price').cast(pl.Float32).fill_null(0.0),
        pl.col('location').cast(pl.Categorical)
    ]).filter(
        (pl.col('event_type') == 'purchased') | (pl.col('quantity') > 0)
    ).with_columns(
        pl.col('event_ts').dt.month().cast(pl.Int8).alias('month')
    ).collect()
    return df

# EDIT THIS PATH TO YOUR KAGGLE DATASET PATH
T_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/transaction_full_2025.parquet'
I_PATH = '/kaggle/input/datasets/kinonquc/qkindataset2/items.parquet'

# Fallback for local testing if paths do not exist
import os
if not os.path.exists(T_PATH):
    T_PATH = '../transaction_full_2025.parquet'
    I_PATH = '../items.parquet'

df_raw = load_and_prep_data(T_PATH)
items_df = pl.read_parquet(I_PATH).with_columns(pl.col('item_id').cast(pl.Utf8))

if 'size' in items_df.columns:
    items_df = items_df.with_columns(
        pl.col('size').map_elements(standardize_age, return_dtype=pl.Float32).alias('item_target_age')
    )
else:
    items_df = items_df.with_columns(pl.lit(-1.0).cast(pl.Float32).alias('item_target_age'))

print(f"Total transactions: {df_raw.height:,}")

# Time Splits
df_train_base = df_raw.filter(pl.col('month') <= 10)
df_val_truth  = df_raw.filter(pl.col('month') == 11)
df_train_val_base = df_raw.filter(pl.col('month') <= 11)
df_test_truth = df_raw.filter(pl.col('month') == 12)

print(f"Train base (Jan-Oct): {df_train_base.height:,}")


Total transactions: 41,470,317
Train base (Jan-Oct): 33,640,568


## 3. Stage 1: Candidate Generation (Retrieval)
Retrieve ~200 candidates per user using SVD, Replenishment, and Popularity.


In [12]:
class SVDRetriever:
    def __init__(self, n_components=64):
        self.model = TruncatedSVD(n_components=n_components, random_state=42)
        
    def fit(self, history_df):
        interactions = history_df.group_by(['customer_id', 'item_id']).agg(pl.col('quantity').sum().alias('weight'))
        self.users = interactions['customer_id'].unique().to_list()
        self.items = interactions['item_id'].unique().to_list()
        self.user2idx = {u: i for i, u in enumerate(self.users)}
        self.item2idx = {i: idx for idx, i in enumerate(self.items)}
        row_idx = [self.user2idx[u] for u in interactions['customer_id']]
        col_idx = [self.item2idx[i] for i in interactions['item_id']]
        
        self.matrix = csr_matrix((interactions['weight'].to_numpy(), (row_idx, col_idx)), shape=(len(self.users), len(self.items)))
        self.user_factors = self.model.fit_transform(self.matrix)
        self.item_factors = self.model.components_.T
        self.pop_items = interactions.group_by('item_id').agg(pl.col('weight').sum()).sort('weight', descending=True)['item_id'].head(100).to_list()
        return self
    
    def retrieve(self, target_users, top_k=100):
        candidates = []
        for u in target_users:
            if u in self.user2idx:
                u_idx = self.user2idx[u]
                scores = self.user_factors[u_idx] @ self.item_factors.T
                top_indices = np.argsort(-scores)[:top_k]
                retrieved = [self.items[i] for i in top_indices]
            else:
                retrieved = self.pop_items[:top_k]
            for i in retrieved:
                candidates.append({'customer_id': u, 'item_id': i})
        # Enforce correct datatypes to match history_df
        return pl.DataFrame(candidates, schema={'customer_id': pl.Int32, 'item_id': pl.Utf8})

def generate_candidates(history_df, target_users, top_k=100):
    svd = SVDRetriever(n_components=32).fit(history_df)
    df_cands_svd = svd.retrieve(target_users, top_k=top_k)
    
    df_rep = history_df.filter(pl.col('customer_id').is_in(target_users)).select(['customer_id', 'item_id']).unique()
    
    pop_items = svd.pop_items[:50]
    pop_cands = pl.DataFrame([{'customer_id': u} for u in target_users], schema={'customer_id': pl.Int32}).join(
        pl.DataFrame({'item_id': pop_items}, schema={'item_id': pl.Utf8}), 
        how='cross'
    )
    
    all_cands = pl.concat([df_cands_svd, df_rep, pop_cands]).unique(subset=['customer_id', 'item_id'])
    return all_cands


## 4. Stage 2: Advanced Feature Engineering (50 Ideas)


In [13]:
def build_features(history_df, candidates_df, items_df):
    # --- User Features ---
    user_feats = history_df.group_by('customer_id').agg([
        pl.col('item_id').n_unique().cast(pl.Float32).alias('user_unique_items'),
        pl.col('quantity').sum().cast(pl.Float32).alias('user_total_volume'),
        (pl.col('item_id').n_unique() / pl.len()).cast(pl.Float32).alias('user_exploration_ratio'),
        pl.col('price').mean().cast(pl.Float32).alias('user_avg_price')
    ])
    
    # --- Item Features ---
    item_feats = history_df.group_by('item_id').agg([
        pl.col('quantity').sum().cast(pl.Float32).alias('item_global_vol'),
        pl.col('customer_id').n_unique().cast(pl.Float32).alias('item_unique_buyers')
    ])
    
    max_date = history_df['event_ts'].max()
    if max_date:
        t1 = max_date - pl.duration(days=14)
        t2 = max_date - pl.duration(days=28)
        momentum_df = history_df.group_by('item_id').agg([
            pl.col('event_ts').filter(pl.col('event_ts') >= t1).len().alias('recent_14d_sales'),
            pl.col('event_ts').filter((pl.col('event_ts') >= t2) & (pl.col('event_ts') < t1)).len().alias('prev_14d_sales')
        ]).with_columns(
            (pl.col('recent_14d_sales') / (pl.col('prev_14d_sales') + 1.0)).cast(pl.Float32).alias('item_momentum_ratio')
        ).select(['item_id', 'item_momentum_ratio'])
        item_feats = item_feats.join(momentum_df, on='item_id', how='left').fill_null(1.0)
    
    # --- Interaction Features ---
    inter_feats = history_df.group_by(['customer_id', 'item_id']).agg([
        pl.col('quantity').sum().cast(pl.Float32).alias('ui_buy_vol'),
        (max_date - pl.col('event_ts').max()).dt.total_days().cast(pl.Int16).alias('ui_days_since_last_buy'),
        (pl.col('event_ts').max() - pl.col('event_ts').min()).dt.total_days().cast(pl.Int16).alias('ui_buy_duration_days')
    ]).with_columns([
        (pl.when(pl.col('ui_days_since_last_buy') >= 22).then(1).otherwise(0)).cast(pl.Int8).alias('ui_is_replenishment_due')
    ])
    
    # Join to candidates
    df_feat = candidates_df.join(user_feats, on='customer_id', how='left')
    df_feat = df_feat.join(item_feats, on='item_id', how='left')
    df_feat = df_feat.join(inter_feats, on=['customer_id', 'item_id'], how='left')
    
    # Add Item Metadata (Size converted to Age)
    df_feat = df_feat.join(items_df.select(['item_id', 'item_target_age']), on='item_id', how='left')
    
    df_feat = df_feat.fill_null(0)
    return df_feat


## 5. Negative Sampling and Dataset Construction


In [14]:
def create_dataset(history_df, truth_df, items_df, sample_users=None, n_negatives=30):
    target_users = truth_df['customer_id'].unique().to_list()
    if sample_users and sample_users < len(target_users):
        np.random.seed(42)
        target_users = list(np.random.choice(target_users, sample_users, replace=False))
        truth_df = truth_df.filter(pl.col('customer_id').is_in(target_users))
    
    candidates = generate_candidates(history_df, target_users, top_k=100)
    
    truth_pairs = truth_df.select(['customer_id', 'item_id']).unique().with_columns(pl.lit(1).cast(pl.Int8).alias('target'))
    dataset = candidates.join(truth_pairs, on=['customer_id', 'item_id'], how='left').fill_null(0)
    
    if n_negatives:
        df_pos = dataset.filter(pl.col('target') == 1)
        df_neg = dataset.filter(pl.col('target') == 0)
        df_neg_sampled = df_neg.group_by('customer_id').map_groups(
            lambda df: df.sample(n=min(len(df), n_negatives), seed=42) if len(df) > 0 else df
        )
        dataset = pl.concat([df_pos, df_neg_sampled]).sample(fraction=1.0, seed=42, shuffle=True)
        
    print(f"Building Features for {dataset.height:,} rows...")
    dataset = build_features(history_df, dataset, items_df)
    return dataset

print("=== BUILDING TRAINING DATA (Eval on Nov) ===")
train_data = create_dataset(df_train_base, df_val_truth, items_df, sample_users=50000, n_negatives=30)

print("\n=== BUILDING TEST DATA (Eval on Dec) ===")
test_data = create_dataset(df_train_val_base, df_test_truth, items_df, sample_users=20000, n_negatives=None)


=== BUILDING TRAINING DATA (Eval on Nov) ===
Building Features for 1,584,895 rows...

=== BUILDING TEST DATA (Eval on Dec) ===
Building Features for 2,481,562 rows...


## 6. Stage 3: Train Ranking Model


In [15]:
feature_cols = [
    'user_unique_items', 'user_total_volume', 'user_exploration_ratio', 'user_avg_price',
    'item_global_vol', 'item_unique_buyers', 'item_momentum_ratio', 'item_target_age',
    'ui_buy_vol', 'ui_days_since_last_buy', 'ui_buy_duration_days', 'ui_is_replenishment_due'
]

X_train = train_data.select(feature_cols).to_pandas()
y_train = train_data['target'].to_pandas()

print("Training LightGBM Ranker...")
lgb_model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, n_jobs=-1, class_weight='balanced')
lgb_model.fit(X_train, y_train)

import pandas as pd
fi = pd.DataFrame({'feature': feature_cols, 'importance': lgb_model.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print("\nTop Features:")
print(fi.head(12))


Training LightGBM Ranker...
[LightGBM] [Info] Number of positive: 84895, number of negative: 1500000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.171248 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2309
[LightGBM] [Info] Number of data points in the train set: 1584895, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000

Top Features:
                    feature  importance
4           item_global_vol        1572
6       item_momentum_ratio        1415
5        item_unique_buyers        1384
3            user_avg_price         984
0         user_unique_items         917
1         user_total_volume         773
9    ui_days_since_last_buy         724
2    user_exploration_ratio         664
8                ui_buy_vol         378
10     ui_buy_duration_days         115
7           item_ta

## 7. Stage 4: Inference and Evaluation


In [16]:
X_test = test_data.select(feature_cols).to_pandas()
test_data = test_data.with_columns(pl.Series(name='pred_score', values=lgb_model.predict_proba(X_test)[:, 1]))

top10_preds = (
    test_data
    .sort(['customer_id', 'pred_score'], descending=[False, True])
    .group_by('customer_id', maintain_order=True)
    .head(10)
)

truth_map = df_test_truth.filter(pl.col('customer_id').is_in(top10_preds['customer_id'].unique().to_list()))
truth_map = truth_map.group_by('customer_id').agg(pl.col('item_id'))
truth_dict = {row[0]: set(row[1]) for row in truth_map.iter_rows()}
pred_map = top10_preds.group_by('customer_id').agg(pl.col('item_id'))
pred_dict = {row[0]: list(row[1]) for row in pred_map.iter_rows()}

def evaluate_metrics(pred_dict, truth_dict):
    total_hits, mrr_sum, p10_sum, iou_sum, map_sum = 0, 0.0, 0.0, 0.0, 0.0
    n_users = len(truth_dict)
    if n_users == 0: return {}
    
    for uid, truth in truth_dict.items():
        preds = pred_dict.get(uid, [])
        hits = [p for p in preds if p in truth]
        total_hits += len(hits)
        
        p10 = len(hits) / 10.0 if len(preds) > 0 else 0.0
        p10_sum += p10
        
        mrr = 0.0
        for i, p in enumerate(preds):
            if p in truth:
                mrr = 1.0 / (i + 1)
                break
        mrr_sum += mrr
        
        intersection = len(set(preds) & truth)
        union = len(set(preds) | truth)
        iou_sum += intersection / union if union > 0 else 0.0
        
        ap_hits, ap_sum = 0, 0.0
        for i, p in enumerate(preds):
            if p in truth:
                ap_hits += 1
                ap_sum += ap_hits / (i + 1)
        map_sum += ap_sum / min(len(truth), 10) if len(truth) > 0 else 0.0
        
    return {
        'Total Correct Hits': total_hits,
        'Precision@10': p10_sum / n_users,
        'MAP': map_sum / n_users,
        'MRR': mrr_sum / n_users,
        'IoU': iou_sum / n_users
    }

metrics = evaluate_metrics(pred_dict, truth_dict)
print("=== FINAL EVALUATION ON DEC 2025 (Test Set) ===")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


=== FINAL EVALUATION ON DEC 2025 (Test Set) ===
Total Correct Hits: 16431
Precision@10: 0.0822
MAP: 0.1859
MRR: 0.3466
IoU: 0.0612
